# Módulo común
Rutas del proyecto y lectura del PDF. Lo usan todos los pasos con `%run -i modulos/comun.ipynb`.

In [ ]:
import re
import subprocess
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent
PDF = RAIZ / "data" / "raw" / "anuario_2023.pdf"
INTERIM = RAIZ / "data" / "interim"
PROCESSED = RAIZ / "data" / "processed"
CURATED = RAIZ / "data" / "curated"
REPORTS = RAIZ / "reports"

CACHE_TEXTO = INTERIM / "texto_crudo.txt"

## Texto plano de las páginas

In [ ]:
# Membretes y pies que se repiten en casi todas las páginas.
RUIDO_INSTITUCIONAL = [
    "Órgano Judicial de Bolivia",
    "Jefatura Nacional de Estudios",
    "DIRECCIÓN NACIONAL DE POLÍTICAS DE GESTIÓN",
    "UNIDAD NACIONAL DE ESTUDIOS TÉCNICOS Y ESTADÍSTICOS",
]
RE_PIE = re.compile(r"Anuario\s+Estad[íi]stico")
# "Consejo de la Magistratura" es membrete, salvo en 14.1.x donde es una fila con datos.
AMBIGUOS = ["Consejo de la Magistratura"]
RE_NUM_SUELTO = re.compile(r"(?<![A-Za-zÁÉÍÓÚÑáéíóúñ])-?\d[\d.,]*%?")


def paginas(usar_cache=True):
    if usar_cache and CACHE_TEXTO.exists():
        crudo = CACHE_TEXTO.read_text(encoding="utf-8", errors="replace")
    else:
        res = subprocess.run(["pdftotext", "-layout", str(PDF), "-"],
                             capture_output=True, text=True, errors="replace")
        if res.returncode != 0:
            raise RuntimeError("pdftotext falló:\n" + res.stderr)
        crudo = res.stdout
        INTERIM.mkdir(parents=True, exist_ok=True)
        CACHE_TEXTO.write_text(crudo, encoding="utf-8")
    pags = crudo.split("\f")
    if len(pags) > 0 and pags[-1] == "":
        pags.pop()
    return pags


def es_ruido(linea):
    for r in RUIDO_INSTITUCIONAL:
        if r in linea:
            return True
    if RE_PIE.search(linea):
        return True
    for a in AMBIGUOS:
        if a in linea:
            return len(RE_NUM_SUELTO.findall(linea)) < 2
    return False


def lineas_utiles(pagina):
    salida = []
    for l in pagina.splitlines():
        if l.strip() and not es_ruido(l):
            salida.append(l.rstrip())
    return salida

## Coordenadas de cada palabra (`pdftotext -bbox-layout`)
Una palabra es `(x_ini, y_ini, x_fin, y_fin, texto)`.

In [ ]:
RE_PALABRA = re.compile(
    r'<word xMin="([\d.]+)" yMin="([\d.]+)" xMax="([\d.]+)" yMax="([\d.]+)">([^<]*)</word>')


def palabras_bbox(pagina):
    res = subprocess.run(["pdftotext", "-bbox-layout", "-f", str(pagina), "-l", str(pagina), str(PDF), "-"],
                         capture_output=True, text=True, errors="replace")
    if res.returncode != 0:
        raise RuntimeError("pdftotext -bbox-layout falló en la página " + str(pagina) + ":\n" + res.stderr)
    palabras = []
    for a, b, c, d, t in RE_PALABRA.findall(res.stdout):
        palabras.append((float(a), float(b), float(c), float(d), t.strip()))
    return palabras


def centro_vertical(caja):
    return (caja[1] + caja[3]) / 2


def agrupar_por_y(cajas, tolerancia=3.5):
    # Dos cajas van a la misma fila si sus centros están a menos de 3,5 puntos
    # (las filas del anuario están a ~8,4). El ancla es el promedio del grupo para
    # que una llamada a nota al pie (voladita, 2,4 puntos más arriba) no abra otra fila.
    filas = []
    for c in sorted(cajas, key=centro_vertical):
        centro = centro_vertical(c)
        if len(filas) > 0 and abs(centro - filas[-1][0]) < tolerancia:
            grupo = filas[-1][1]
            filas[-1][0] = (filas[-1][0] * len(grupo) + centro) / (len(grupo) + 1)
            grupo.append(c)
        else:
            filas.append([centro, [c]])
    salida = []
    for centro, grupo in filas:
        salida.append(sorted(grupo, key=lambda c: c[0]))
    return salida


def filas_bbox(pagina, tolerancia=3.5):
    return agrupar_por_y(palabras_bbox(pagina), tolerancia)


def agrupar_por_x(rangos):
    # Fusiona rangos horizontales [ini, fin) que se solapan: cada resultado es una columna.
    columnas = []
    for ini, fin in sorted(rangos):
        if len(columnas) > 0 and ini < columnas[-1][1]:
            columnas[-1][1] = max(columnas[-1][1], fin)
        else:
            columnas.append([ini, fin])
    return columnas

## Bloques y celdas
`<block>` agrupa las líneas de una misma celda del PDF (resuelve rótulos partidos en
varias líneas); `<line>` es una línea dentro de la celda.
Bloque = `(x_ini, y_ini, x_fin, y_fin, texto, lineas)`; línea = `(x_ini, y_ini, x_fin, y_fin, texto, palabras)`.
Se vuelca un tramo de páginas de una vez y se guarda en memoria.

In [ ]:
RE_PAGINA_BBOX = re.compile(r'<page width="[\d.]+" height="[\d.]+">(.*?)</page>', re.S)
RE_BLOQUE_BBOX = re.compile(
    r'<block xMin="([\d.]+)" yMin="([\d.]+)" xMax="([\d.]+)" yMax="([\d.]+)">(.*?)</block>', re.S)
RE_LINEA_BBOX = re.compile(
    r'<line xMin="([\d.]+)" yMin="([\d.]+)" xMax="([\d.]+)" yMax="([\d.]+)">(.*?)</line>', re.S)

cache_bloques = {}


def caja_de(hijos):
    x0 = min(h[0] for h in hijos)
    y0 = min(h[1] for h in hijos)
    x1 = max(h[2] for h in hijos)
    y1 = max(h[3] for h in hijos)
    return (x0, y0, x1, y1)


def bloques_bbox(desde, hasta=None):
    if hasta is None:
        hasta = desde
    faltan = []
    for p in range(desde, hasta + 1):
        if p not in cache_bloques:
            faltan.append(p)
    if len(faltan) > 0:
        ini = min(faltan)
        fin = max(faltan)
        res = subprocess.run(["pdftotext", "-bbox-layout", "-f", str(ini), "-l", str(fin), str(PDF), "-"],
                             capture_output=True, text=True, errors="replace")
        if res.returncode != 0:
            raise RuntimeError("pdftotext -bbox-layout falló en " + str(ini) + "-" + str(fin) + ":\n" + res.stderr)
        n = ini
        for mp in RE_PAGINA_BBOX.finditer(res.stdout):
            bloques = []
            for mb in RE_BLOQUE_BBOX.finditer(mp.group(1)):
                lineas = []
                for ml in RE_LINEA_BBOX.finditer(mb.group(5)):
                    ws = []
                    for a, b, c, d, t in RE_PALABRA.findall(ml.group(5)):
                        if t.strip():
                            ws.append((float(a), float(b), float(c), float(d), t.strip()))
                    if len(ws) > 0:
                        caja = caja_de(ws)
                        texto = " ".join(w[4] for w in ws)
                        lineas.append((caja[0], caja[1], caja[2], caja[3], texto, ws))
                if len(lineas) > 0:
                    caja = caja_de(lineas)
                    texto = " ".join(l[4] for l in lineas)
                    bloques.append((caja[0], caja[1], caja[2], caja[3], texto, lineas))
            cache_bloques[n] = bloques
            n = n + 1
    salida = {}
    for p in range(desde, hasta + 1):
        salida[p] = cache_bloques[p]
    return salida


def celdas_bbox(desde, hasta=None):
    salida = {}
    for p, bloques in bloques_bbox(desde, hasta).items():
        celdas = []
        for b in bloques:
            for linea in b[5]:
                celdas.append(linea)
        salida[p] = celdas
    return salida